# Day 1


## Phase 1

In [1]:
import pandas as pd
import kagglehub
from sqlalchemy import create_engine, text

# 1. Download the latest version of the Netflix dataset
path = kagglehub.dataset_download("shivamb/netflix-shows")
csv_path = f"{path}/netflix_titles.csv"

# 2. Load the dataset into Pandas
df_netflix = pd.read_csv(csv_path)

# 3. Create our production SQL engine
engine = create_engine('sqlite:///netflix_warehouse.db')

print(" Real-World Netflix Dataset Ingested Successfully!")
print(f"Total Rows: {df_netflix.shape[0]} | Total Columns: {df_netflix.shape[1]}")

Using Colab cache for faster access to the 'netflix-shows' dataset.
 Real-World Netflix Dataset Ingested Successfully!
Total Rows: 8807 | Total Columns: 12


In [2]:
df_netflix.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [3]:
df_netflix.dtypes

,0
show_id,object
type,object
title,object
director,object
cast,object
country,object
date_added,object
release_year,int64
rating,object
duration,object


**Step 1 Vectorized Text Alignment: Clean up the type and title columns by chaining .str.strip() to remove accidental spacing padding.**

In [4]:
df_netflix['type'] = df_netflix['type'].str.strip()
df_netflix['title'] = df_netflix['title'].str.strip()

print(df_netflix[['type', 'title']])

         type                  title
0       Movie   Dick Johnson Is Dead
1     TV Show          Blood & Water
2     TV Show              Ganglands
3     TV Show  Jailbirds New Orleans
4     TV Show           Kota Factory
...       ...                    ...
8802    Movie                 Zodiac
8803  TV Show            Zombie Dumb
8804    Movie             Zombieland
8805    Movie                   Zoom
8806    Movie                 Zubaan

[8807 rows x 2 columns]


**Step 2: Deduplication & Handling Empty Values**

In [5]:
df_netflix = df_netflix.drop_duplicates(subset=['show_id'])
df_netflix['show_id']

,show_id
0,s1
1,s2
2,s3
3,s4
4,s5
...,...
8802,s8803
8803,s8804
8804,s8805
8805,s8806


In [6]:
df_netflix = df_netflix.fillna({
    'director': 'Unknown',
    'country': 'Unknown'
    })

In [7]:
df_netflix[['director','country']]

,director,country
0,Kirsten Johnson,United States
1,Unknown,South Africa
2,Julien Leclercq,Unknown
3,Unknown,Unknown
4,Unknown,India
...,...,...
8802,David Fincher,United States
8803,Unknown,Unknown
8804,Ruben Fleischer,United States
8805,Peter Hewitt,United States


In [8]:
print(len(df_netflix))

8807


**Step 3: The Production SQL Warehouse Sync**

In [9]:
with engine.connect() as conn:
  # Write the DataFrame to the SQL database
  df_netflix.to_sql('netflix_shows', con=conn, if_exists='replace', index=False)

  query = "SELECT type, title, director FROM netflix_shows LIMIT 3;"
  result = conn.execute(text(query)).fetchall()
  print("--- STORAGE VERIFICATION ---")
  for row in result:
    print(row)

--- STORAGE VERIFICATION ---
('Movie', 'Dick Johnson Is Dead', 'Kirsten Johnson')
('TV Show', 'Blood & Water', 'Unknown')
('TV Show', 'Ganglands', 'Julien Leclercq')


**Step 4: Extracting Meaningful Business Insights**

In [10]:
import pandas as pd

query = """ SELECT type, COUNT(*) as total_count FROM netflix_shows GROUP BY type;"""
insight_df = pd.read_sql(query, engine)
print("--- 🍿 NETFLIX CONTENT SPLIT REPORT ---")
print(insight_df)

--- 🍿 NETFLIX CONTENT SPLIT REPORT ---
      type  total_count
0    Movie         6131
1  TV Show         2676


**Step 5: The Elite Director Leaderboard**

In [11]:
query = """ SELECT director, COUNT(*) as movie_count FROM netflix_shows WHERE type = 'Movie' GROUP BY director ORDER BY movie_count DESC LIMIT 10;"""
insight_df = pd.read_sql(query, engine)
print("--- TOP 10 DIRECTORS WITH THE MOST MOVIES ---")
print(insight_df)

--- TOP 10 DIRECTORS WITH THE MOST MOVIES ---
                 director  movie_count
0                 Unknown          188
1           Rajiv Chilaka           19
2  Raúl Campos, Jan Suter           18
3             Suhas Kadav           16
4            Marcus Raboy           15
5               Jay Karas           14
6     Cathy Garcia-Molina           13
7         Youssef Chahine           12
8         Martin Scorsese           12
9             Jay Chapman           12


In [12]:
import pandas as pd

query = """
SELECT director,
       COUNT(*) as movie_count
FROM netflix_shows
WHERE type = 'Movie' AND director != 'Unknown'
GROUP BY director
ORDER BY movie_count DESC
LIMIT 10;
"""

insight_df = pd.read_sql(query, engine)
print("--- 🏆 THE REAL TOP 10 NAMED DIRECTORS 🏆 ---")
print(insight_df)

--- 🏆 THE REAL TOP 10 NAMED DIRECTORS 🏆 ---
                 director  movie_count
0           Rajiv Chilaka           19
1  Raúl Campos, Jan Suter           18
2             Suhas Kadav           16
3            Marcus Raboy           15
4               Jay Karas           14
5     Cathy Garcia-Molina           13
6         Youssef Chahine           12
7         Martin Scorsese           12
8             Jay Chapman           12
9        Steven Spielberg           11


## Phase 2 — Mastering Groupby Aggregations & Analytical Transformations!

In [13]:
average_years = df_netflix.groupby('type')['release_year'].mean()
print("NETFLIX AVERAGE YEARS")
print(average_years)

NETFLIX AVERAGE YEARS
type
Movie      2013.121514
TV Show    2016.605755
Name: release_year, dtype: float64


In [14]:
df_netflix.groupby(['type', 'rating'])['show_id'].count()

type     rating  
Movie    66 min         1
         74 min         1
         84 min         1
         G             41
         NC-17          3
         NR            75
         PG           287
         PG-13        490
         R            797
         TV-14       1427
         TV-G         126
         TV-MA       2062
         TV-PG        540
         TV-Y         131
         TV-Y7        139
         TV-Y7-FV       5
         UR             3
TV Show  NR             5
         R              2
         TV-14        733
         TV-G          94
         TV-MA       1145
         TV-PG        323
         TV-Y         176
         TV-Y7        195
         TV-Y7-FV       1
Name: show_id, dtype: int64

In [15]:
year_summary = df_netflix.groupby('type')['release_year'].agg(['count', 'max', 'min'])

In [16]:
print("NETFLIX YEAR SUMMARY")
print(year_summary)

NETFLIX YEAR SUMMARY
         count   max   min
type                      
Movie     6131  2021  1942
TV Show   2676  2021  1925


In [17]:
top_countires = df_netflix.groupby('country')['show_id'].count()
having_filter = top_countires[top_countires > 200]
print("NETFLIX TOP COUNTRIES")
print(having_filter)

NETFLIX TOP COUNTRIES
country
India              972
Japan              245
United Kingdom     419
United States     2818
Unknown            831
Name: show_id, dtype: int64


In [18]:
# Create a new column with global counts per type
df_netflix['type_total_count'] = df_netflix.groupby('type')['show_id'].transform('count')

# Verify the result by printing top 3 rows
print(df_netflix[['type', 'title', 'type_total_count']].head(3))


      type                 title  type_total_count
0    Movie  Dick Johnson Is Dead              6131
1  TV Show         Blood & Water              2676
2  TV Show             Ganglands              2676


In [19]:
country_total_titles = df_netflix.groupby('country')['show_id'].count()
having_filter = country_total_titles[(country_total_titles > 100) & (country_total_titles.index != 'Unknown')]
df_hubs = df_netflix.groupby(['country', 'type'])['show_id'].agg(['count', 'nunique'])
df_hubs = df_hubs.reset_index()
df_hubs = df_hubs[df_hubs['country'].isin(having_filter.index)]

# Filter df_netflix to include only countries that are in the having_filter
filtered_netflix_for_hubs = df_netflix[df_netflix['country'].isin(having_filter.index)]

# Now, group this filtered DataFrame by country and type, and aggregate release_year
hub_report = filtered_netflix_for_hubs.groupby(['country', 'type'])['release_year'].agg(['min', 'max', 'count'])

# Print the final summary matrix
print(hub_report)

                         min   max  count
country        type                      
Canada         Movie    1998  2020    122
               TV Show  1998  2021     59
Egypt          Movie    1954  2020     92
               TV Show  2015  2021     14
France         Movie    1974  2021     75
               TV Show  2002  2021     49
India          Movie    1959  2021    893
               TV Show  2009  2021     79
Japan          Movie    1979  2021     76
               TV Show  1981  2021    169
Mexico         Movie    1991  2021     70
               TV Show  1979  2021     40
South Korea    Movie    2004  2021     41
               TV Show  2009  2021    158
Spain          Movie    2008  2021     97
               TV Show  2013  2021     48
Turkey         Movie    1999  2021     76
               TV Show  2005  2021     29
United Kingdom Movie    1975  2021    206
               TV Show  1974  2021    213
United States  Movie    1942  2021   2058
               TV Show  1945  2021

# Day 2

## Phase 3 — Data Quality Audit

### Task 1: Null Values
To identify and analyze missing data in your dataset, chain the missing value detection and summation methods together. You can filter out the columns with clean data by slicing the final Series.

* **Step 1:** Use `isna()` or `isnull()` to detect missing values across the DataFrame.
* **Step 2:** Chain the `.sum()` method to count the total nulls per column.
* **Step 3:** Slice the resulting Series to only display columns where the count is greater than 0 (`> 0`).

---

### Task 2: Full Duplicates
Pandas provides a straightforward way to check for rows that are completely identical across all columns.

* **Step 1:** Call the built-in `.duplicated()` method on your DataFrame. By leaving the subset parameter blank, Pandas will automatically evaluate the entire row.
* **Step 2:** Append `.sum()` to the method call to get the total count of duplicate rows.

---

### Task 3: Future Dates
To ensure data integrity against futuristic or erroneous entries, you can dynamically validate the `release_year` column against the current calendar year.

* **Step 1:** Import the `datetime` module:
  ```python
  from datetime import datetime

In [20]:
# Step 1 & 2: Detect nulls and sum them up per column
null_counts = df_netflix.isnull().sum()

# Step 3: Slice the Series to only show columns with more than 0 nulls
missing_data_columns = null_counts[null_counts > 0]

# Display the result
missing_data_columns

,0
cast,825
date_added,10
rating,4
duration,3


In [21]:
duplicates_data = df_netflix.duplicated().sum()
duplicates_data

np.int64(0)

In [22]:
from datetime import datetime
current_year = datetime.now().year
future_shows = df_netflix[df_netflix['release_year'] > current_year]
len(future_shows)

0

In [23]:
df_netflix.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,type_total_count
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",6131
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",2676
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,2676
3,s4,TV Show,Jailbirds New Orleans,Unknown,NaN,Unknown,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",2676
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,2676


In [24]:
# 1. Convert the 'date_added' column and turn bad text into NaT (null)
df_netflix['date_added'] = pd.to_datetime(df_netflix['date_added'], errors='coerce')

# 2. Store the count of failed conversions into your variable name
failed_dates_count = df_netflix['date_added'].isna().sum()

# 3. Print the variable to see the result
print(failed_dates_count)

98


In [25]:
df_netflix['date_added'].head()

,date_added
0,2021-09-25
1,2021-09-24
2,2021-09-24
3,2021-09-24
4,2021-09-24


In [26]:
df_netflix['month_added'] = df_netflix['date_added'].dt.month
df_netflix['year_added'] = df_netflix['date_added'].dt.year
df_netflix['date_added'] = df_netflix['date_added'].dt.date

#Check your handy work by looking at the new columns
df_netflix[['date_added', 'year_added', 'month_added']].head()

,date_added,year_added,month_added
0,2021-09-25,2021.0,9.0
1,2021-09-24,2021.0,9.0
2,2021-09-24,2021.0,9.0
3,2021-09-24,2021.0,9.0
4,2021-09-24,2021.0,9.0


In [28]:
yearly_report = df_netflix.groupby('year_added')['show_id'].count()
yearly_report

,show_id
year_added,
2008.0,2
2009.0,2
2010.0,1
2011.0,13
2012.0,3
2013.0,10
2014.0,23
2015.0,73
2016.0,418


In [32]:
# 1. Split the text column by space into two new columns
df_netflix[['duration_num', 'duration_unit']] = df_netflix['duration'].str.split(' ', expand=True)

# 2. Convert the number column from text to numeric using .astype()
df_netflix['duration_num'] = df_netflix['duration_num'].astype(float)

# 3. Verify it worked by checking the new columns
df_netflix[['duration', 'duration_num', 'duration_unit']].head()

,duration,duration_num,duration_unit
0,90 min,90.0,min
1,2 Seasons,2.0,Seasons
2,1 Season,1.0,Season
3,1 Season,1.0,Season
4,2 Seasons,2.0,Seasons


In [33]:
df_netflix.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,type_total_count,month_added,year_added,duration_num,duration_unit
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",6131,9.0,2021.0,90.0,min
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",2676,9.0,2021.0,2.0,Seasons
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,2676,9.0,2021.0,1.0,Season
3,s4,TV Show,Jailbirds New Orleans,Unknown,NaN,Unknown,2021-09-24,2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",2676,9.0,2021.0,1.0,Season
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,2676,9.0,2021.0,2.0,Seasons


In [38]:
# Filter for Movies, pick the duration_num column, and find the mean
avg_movie_duration = df_netflix[df_netflix['type'] == 'Movie']['duration_num'].mean()

# Print the result
print(f"Average Movie Duration: {avg_movie_duration} minutes")

Average Movie Duration: 99.57718668407311 minutes


In [39]:
# Save your fully cleaned data to a brand new SQL table
df_netflix.to_sql('clean_netflix_shows', con=engine, if_exists='replace', index=False)

print("🎉 Data Cleaning Complete! Cleaned dataset synced to production SQL warehouse.")

🎉 Data Cleaning Complete! Cleaned dataset synced to production SQL warehouse.


In [40]:
from sqlalchemy import select, func, desc, table, column

# Define a quick structural reference to your SQL table columns
# (This acts as a map for SQLAlchemy Core to build the query)
shows_tbl = table('netflix_shows',
                  column('director'),
                  column('type'))

# --- THE BLUEPRINT EXPRESSION ---
# Task: Build the top named directors query using Python objects
stmt = (
    select(
        shows_tbl.c.director,
        func.count(shows_tbl.c.director).label('movie_count')
    )
    .where(shows_tbl.c.type == 'Movie')
    .where(shows_tbl.c.director != 'Unknown')
    .group_by(shows_tbl.c.director)
    .order_by(desc('movie_count'))
    .limit(10)
)

# Execute using pandas read_sql to keep our clean DataFrame output
core_leaderboard_df = pd.read_sql(stmt, engine)

print("--- TOP 10 DIRECTORS (SQLALCHEMY CORE) ---")
print(core_leaderboard_df)

--- TOP 10 DIRECTORS (SQLALCHEMY CORE) ---
                 director  movie_count
0           Rajiv Chilaka           19
1  Raúl Campos, Jan Suter           18
2             Suhas Kadav           16
3            Marcus Raboy           15
4               Jay Karas           14
5     Cathy Garcia-Molina           13
6         Youssef Chahine           12
7         Martin Scorsese           12
8             Jay Chapman           12
9        Steven Spielberg           11


In [42]:
import pandas as pd
import kagglehub
from sqlalchemy import create_engine

def extract():
    """Extract Phase: Ingest raw data from the external source."""
    print(" [1/3] Extracting raw data from Kaggle...")
    path = kagglehub.dataset_download("shivamb/netflix-shows")
    csv_path = f"{path}/netflix_titles.csv"

    raw_df = pd.read_csv(csv_path)
    return raw_df


def transform(df):
    """Transform Phase: Clean data types, handle nulls, and parse strings."""
    print(" [2/3] Transforming and sanitizing dataset...")

    # Create a local copy to avoid mutating the input argument
    working_df = df.copy()

    # 1. String Padding Cleanup
    working_df['type'] = working_df['type'].str.strip()
    working_df['title'] = working_df['title'].str.strip()

    # 2. Handle missing metadata safely
    working_df = working_df.fillna({'director': 'Unknown', 'country': 'Unknown'})

    # 3. Securely handle dates and fix the Float-Conversion gap using Int64
    working_df['date_added'] = pd.to_datetime(working_df['date_added'], errors='coerce')
    working_df['year_added'] = working_df['date_added'].dt.year.astype('Int64')
    working_df['month_added'] = working_df['date_added'].dt.month.astype('Int64')
    working_df['date_added'] = working_df['date_added'].dt.date

    # 4. Parse messy duration column
    working_df[['duration_num', 'duration_unit']] = working_df['duration'].str.split(' ', expand=True)
    working_df['duration_num'] = working_df['duration_num'].astype(float)

    return working_df


def load(df, db_uri='sqlite:///netflix_warehouse.db'):
    """Load Phase: Securely sink production data to the warehouse database."""
    print(f" [3/3] Loading target data into warehouse...")
    local_engine = create_engine(db_uri)

    with local_engine.connect() as conn:
        df.to_sql('clean_netflix_shows', con=conn, if_exists='replace', index=False)

    print(" ETL Execution Complete! Warehouse table 'clean_netflix_shows' is live.")


# --- AIRFLOW-STYLE EXECUTION ENTRYPOINT ---
if __name__ == "__main__":
    # Data flows cleanly from one functional scope to the next
    extracted_data = extract()
    transformed_data = transform(extracted_data)
    load(transformed_data)

 [1/3] Extracting raw data from Kaggle...
Using Colab cache for faster access to the 'netflix-shows' dataset.
 [2/3] Transforming and sanitizing dataset...
 [3/3] Loading target data into warehouse...
 ETL Execution Complete! Warehouse table 'clean_netflix_shows' is live.


In [43]:
import pandas as pd
import kagglehub
from sqlalchemy import create_engine, text

def extract():
    """Extract Phase: Ingest raw data from the external source."""
    print("⏳ [1/3] Extracting raw data from Kaggle...")
    path = kagglehub.dataset_download("shivamb/netflix-shows")
    csv_path = f"{path}/netflix_titles.csv"

    raw_df = pd.read_csv(csv_path)
    return raw_df


def transform(df):
    """Transform Phase: Clean data types, handle nulls, and parse strings."""
    print("🧹 [2/3] Transforming and sanitizing dataset...")

    # --- TASK 14: SIMULATE TASK FAILURE HANDLING ---
    # Catch empty data drops from network/API glitches before wasting compute
    if df.empty:
        raise ValueError("❌ Upstream Extraction Failure: Received an empty DataFrame at transformation entrypoint!")

    working_df = df.copy()

    # 1. String Padding Cleanup
    working_df['type'] = working_df['type'].str.strip()
    working_df['title'] = working_df['title'].str.strip()

    # 2. Handle missing metadata safely
    working_df = working_df.fillna({'director': 'Unknown', 'country': 'Unknown'})

    # 3. Handle dates and use Int64 type
    working_df['date_added'] = pd.to_datetime(working_df['date_added'], errors='coerce')
    working_df['year_added'] = working_df['date_added'].dt.year.astype('Int64')
    working_df['month_added'] = working_df['date_added'].dt.month.astype('Int64')
    working_df['date_added'] = working_df['date_added'].dt.date

    # 4. Parse messy duration column
    working_df[['duration_num', 'duration_unit']] = working_df['duration'].str.split(' ', expand=True)
    working_df['duration_num'] = working_df['duration_num'].astype(float)

    return working_df


def load(df, db_uri='sqlite:///netflix_warehouse.db'):
    """Load Phase: Securely sink production data to the warehouse database."""
    print(f"🚀 [3/3] Loading target data into warehouse...")
    local_engine = create_engine(db_uri)

    with local_engine.connect() as conn:
        # Idempotent write: 'replace' ensures we never duplicate records if re-run
        df.to_sql('clean_netflix_shows', con=conn, if_exists='replace', index=False)

        # --- TASK 13: VERIFIABLE AUDIT ---
        # Query back the exact count straight from the physical storage layer
        result = conn.execute(text("SELECT COUNT(*) FROM clean_netflix_shows")).scalar()

    print("🎉 ETL Execution Complete!")
    return result


# --- AIRFLOW-STYLE EXECUTION ENTRYPOINT ---
if __name__ == "__main__":
    # Orchestrate pipeline tasks sequentially
    extracted_data = extract()
    transformed_data = transform(extracted_data)

    # Capture the verification metric returned from the database load
    committed_rows = load(transformed_data)

    print("\n--- 📊 PRODUCTION DATA VERIFICATION REPORT ---")
    print(f"Processed Rows in Memory: {len(transformed_data)}")
    print(f"Committed Rows in Database: {committed_rows}")

    if len(transformed_data) == committed_rows:
        print("✅ SUCCESS: Data pipeline audit passed! Source matches destination perfectly.")
    else:
        print("❌ WARNING: Data discrepancy detected between memory processing and disk write.")

⏳ [1/3] Extracting raw data from Kaggle...
Using Colab cache for faster access to the 'netflix-shows' dataset.
🧹 [2/3] Transforming and sanitizing dataset...
🚀 [3/3] Loading target data into warehouse...
🎉 ETL Execution Complete!

--- 📊 PRODUCTION DATA VERIFICATION REPORT ---
Processed Rows in Memory: 8807
Committed Rows in Database: 8807
✅ SUCCESS: Data pipeline audit passed! Source matches destination perfectly.


In [2]:
pip install apache-airflow

In [1]:
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime
import netflix_pipeline as pipeline

with DAG(
    dag_id='netflix_warehouse_etl',
    start_date=datetime(2026, 1, 1),
    schedule_interval='@daily',
    catchup=False
) as dag:

    # PythonOperator acts as a wrapper around your decoupled python logic
    extract_task = PythonOperator(
        task_id='extract_raw_kaggle_data',
        python_callable=pipeline.extract
    )

    transform_task = PythonOperator(
        task_id='clean_and_sanitize_metadata',
        python_callable=pipeline.transform
    )

    load_task = PythonOperator(
        task_id='load_to_sqlite_warehouse',
        python_callable=pipeline.load
    )

    # Define the dependency lineage
    extract_task >> transform_task >> load_task

2026-06-17T14:13:07.966935Z [warning  ] The `airflow.operators.python.PythonOperator` attribute is deprecated. Please use `'airflow.providers.standard.operators.python.PythonOperator'`. [py.warnings] category=DeprecatedImportWarning filename=/tmp/ipykernel_27556/2671054101.py lineno=2


ModuleNotFoundError: No module named 'netflix_pipeline'

In [3]:
%%writefile netflix_pipeline.py

import pandas as pd
import kagglehub
from sqlalchemy import create_engine, text

def extract():
    """Extract Phase: Ingest raw data from the external source."""
    print("⏳ [1/3] Extracting raw data from Kaggle...")
    path = kagglehub.dataset_download("shivamb/netflix-shows")
    csv_path = f"{path}/netflix_titles.csv"

    raw_df = pd.read_csv(csv_path)
    return raw_df


def transform(df):
    """Transform Phase: Clean data types, handle nulls, and parse strings."""
    print("🧹 [2/3] Transforming and sanitizing dataset...")

    # --- TASK 14: SIMULATE TASK FAILURE HANDLING ---
    # Catch empty data drops from network/API glitches before wasting compute
    if df.empty:
        raise ValueError("❌ Upstream Extraction Failure: Received an empty DataFrame at transformation entrypoint!")

    working_df = df.copy()

    # 1. String Padding Cleanup
    working_df['type'] = working_df['type'].str.strip()
    working_df['title'] = working_df['title'].str.strip()

    # 2. Handle missing metadata safely
    working_df = working_df.fillna({'director': 'Unknown', 'country': 'Unknown'})

    # 3. Handle dates and use Int64 type
    working_df['date_added'] = pd.to_datetime(working_df['date_added'], errors='coerce')
    working_df['year_added'] = working_df['date_added'].dt.year.astype('Int64')
    working_df['month_added'] = working_df['date_added'].dt.month.astype('Int64')
    working_df['date_added'] = working_df['date_added'].dt.date

    # 4. Parse messy duration column
    working_df[['duration_num', 'duration_unit']] = working_df['duration'].str.split(' ', expand=True)
    working_df['duration_num'] = working_df['duration_num'].astype(float)

    return working_df


def load(df, db_uri='sqlite:///netflix_warehouse.db'):
    """Load Phase: Securely sink production data to the warehouse database."""
    print(f"🚀 [3/3] Loading target data into warehouse...")
    local_engine = create_engine(db_uri)

    with local_engine.connect() as conn:
        # Idempotent write: 'replace' ensures we never duplicate records if re-run
        df.to_sql('clean_netflix_shows', con=conn, if_exists='replace', index=False)

        # --- TASK 13: VERIFIABLE AUDIT ---
        # Query back the exact count straight from the physical storage layer
        result = conn.execute(text("SELECT COUNT(*) FROM clean_netflix_shows")).scalar()

    print("🎉 ETL Execution Complete!")
    return result


# --- AIRFLOW-STYLE EXECUTION ENTRYPOINT ---
if __name__ == "__main__":
    # Orchestrate pipeline tasks sequentially
    extracted_data = extract()
    transformed_data = transform(extracted_data)

    # Capture the verification metric returned from the database load
    committed_rows = load(transformed_data)

    print("\n--- 📊 PRODUCTION DATA VERIFICATION REPORT ---")
    print(f"Processed Rows in Memory: {len(transformed_data)}")
    print(f"Committed Rows in Database: {committed_rows}")

    if len(transformed_data) == committed_rows:
        print("✅ SUCCESS: Data pipeline audit passed! Source matches destination perfectly.")
    else:
        print("❌ WARNING: Data discrepancy detected between memory processing and disk write.")

Writing netflix_pipeline.py
